# Feature Engineering
**Project**: House Price Prediction  
**Notebook**: `03_feature_engineering.ipynb`

In [1]:
import os
import pandas as pd
import numpy as np

data_path = os.path.join("..", "data", "raw", "House_Prices.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "raw", "House_Prices.csv")

df = pd.read_csv(data_path).iloc[:1460].copy()
df = df[df["GrLivArea"] < 4000].reset_index(drop=True)
print("Raw data loaded for feature engineering. Shape:", df.shape)

Raw data loaded for feature engineering. Shape: (1456, 81)


## 1. Feature 1: Total House Area (`TotalArea`)
Sums Basement, 1st Floor, and 2nd Floor square footage.

In [2]:
df['TotalBsmtSF'] = df['TotalBsmtSF'].fillna(0)
df['1stFlrSF'] = df['1stFlrSF'].fillna(0)
df['2ndFlrSF'] = df['2ndFlrSF'].fillna(0)
df['TotalArea'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

print("TotalArea summary:")
print(df['TotalArea'].describe())

TotalArea summary:
count    1456.000000
mean     2551.300137
std       758.353116
min       334.000000
25%      2008.000000
50%      2472.000000
75%      2999.250000
max      6428.000000
Name: TotalArea, dtype: float64


## 2. Feature 2: Total Bathrooms (`TotalBathrooms`)
Weights half baths at 0.5.

In [3]:
df['FullBath'] = df['FullBath'].fillna(0)
df['HalfBath'] = df['HalfBath'].fillna(0)
df['BsmtFullBath'] = df['BsmtFullBath'].fillna(0)
df['BsmtHalfBath'] = df['BsmtHalfBath'].fillna(0)

df['TotalBathrooms'] = (
    df['FullBath'] + (0.5 * df['HalfBath']) + 
    df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
)

print("TotalBathrooms summary:")
print(df['TotalBathrooms'].describe())

TotalBathrooms summary:
count    1456.000000
mean        2.204670
std         0.778141
min         1.000000
25%         2.000000
50%         2.000000
75%         2.500000
max         6.000000
Name: TotalBathrooms, dtype: float64


## 3. Feature 3 & 4: House Age (`HouseAge`) and Remodel Age (`YearsSinceRemodel`)

In [4]:
df['HouseAge'] = np.maximum(0, df['YrSold'] - df['YearBuilt'])
df['YearsSinceRemodel'] = np.maximum(0, df['YrSold'] - df['YearRemodAdd'])

print("HouseAge & YearsSinceRemodel summary:")
print(df[['HouseAge', 'YearsSinceRemodel']].describe())

HouseAge & YearsSinceRemodel summary:
          HouseAge  YearsSinceRemodel
count  1456.000000        1456.000000
mean     36.631868          22.997940
std      30.247555          20.646276
min       0.000000           0.000000
25%       8.000000           4.000000
50%      35.000000          14.000000
75%      54.000000          41.000000
max     136.000000          60.000000


## 4. Feature Verification & Correlation with `SalePrice`

In [5]:
engineered_cols = ['TotalArea', 'TotalBathrooms', 'HouseAge', 'YearsSinceRemodel', 'SalePrice']
corr = df[engineered_cols].corr()

print("Correlation of Engineered Features with SalePrice:")
print(corr['SalePrice'].sort_values(ascending=False))

Correlation of Engineered Features with SalePrice:
SalePrice            1.000000
TotalArea            0.825066
TotalBathrooms       0.635939
YearsSinceRemodel   -0.523102
HouseAge            -0.535507
Name: SalePrice, dtype: float64
